In [ ]:
import numpy as np

# =========================
# PRECISION CONTROL
# =========================
DT = np.longdouble


def asDT(x):
    return np.asarray(x, dtype=DT)

# =========================
# SETTINGS
# =========================
R = DT("8.314")
T = DT("1000.0")

# MQMQA ideal-term exponents (toy defaults)
phi = DT("1.0")
psi = DT("1.0")

# FD step sweep
H_LIST = [DT("1e-2"), DT("1e-3"), DT("1e-4"), DT("1e-5"), DT("1e-6")]

# Two random test points (1–2 requested). Add more seeds if you want.
SEEDS = [0, 1]

# =========================
# VARIABLE ORDER (9 quadruplets)
# =========================
# Order: by anion-pair blocks (XX, XY, YY)
vars_ = [
    "AA_XX","AB_XX","BB_XX",
    "AA_XY","AB_XY","BB_XY",
    "AA_YY","AB_YY","BB_YY",
]
idx = {v:i for i,v in enumerate(vars_)}
M = len(vars_)

cations = ["A","B"]
anions  = ["X","Y"]

def delta(a,b): return DT("1.0") if a == b else DT("0.0")

def parse(v):
    cat, an = v.split("_")
    i,j = cat[0], cat[1]
    k,l = an[0],  an[1]
    return i,j,k,l

quad = [parse(v) for v in vars_]  # list of (i,j,k,l)
ONES = np.ones(M, dtype=DT)

# =========================
# WEIGHT VECTORS (linear maps from quadruplet moles -> derived counts)
# =========================
# site moles on sublattice 1 (cations): n_i = sum_r n_r * (δ_ai+δ_bi)/2
w_site1 = {
    i: np.array([(delta(a,i)+delta(b,i))/DT("2.0") for (a,b,x,y) in quad], dtype=DT)
    for i in cations
}
# site moles on sublattice 2 (anions): n_k = sum_r n_r * (δ_xk+δ_yk)/2
w_site2 = {
    k: np.array([(delta(x,k)+delta(y,k))/DT("2.0") for (a,b,x,y) in quad], dtype=DT)
    for k in anions
}
# pair moles: n_{i/k} = sum_r n_r * ((δ_ai+δ_bi)(δ_xk+δ_yk))/4
w_pair = {
    (i,k): np.array([((delta(a,i)+delta(b,i))*(delta(x,k)+delta(y,k)))/DT("4.0")
                     for (a,b,x,y) in quad], dtype=DT)
    for i in cations for k in anions
}

# row/col sums for F factors
w_row = {i: sum(w_pair[(i,k)] for k in anions) for i in cations}  # A_i
w_col = {k: sum(w_pair[(i,k)] for i in cations) for k in anions}  # B_k
w_Npair = sum(w_pair[(i,k)] for i in cations for k in anions)     # should equal ONES

# =========================
# HELPER DERIVS
# =========================
def ln_linear_derivs(w, n):
    """L = w·n. Return ln(L), (ln L)_{,p} vector, (ln L)_{,pq} matrix."""
    n = asDT(n)
    L = np.dot(w, n)
    if L <= 0:
        raise ValueError("Nonpositive linear form inside log.")
    ln_val = np.log(L)
    ln_p   = w / L
    ln_pq  = -np.outer(w, w) / (L*L)
    return ln_val, ln_p, ln_pq

def ln_ratio_derivs(wA, wB, n):
    """Return derivatives of ln(A/B) with A=wA·n, B=wB·n."""
    n = asDT(n)
    A = np.dot(wA, n)
    B = np.dot(wB, n)
    if A <= 0 or B <= 0:
        raise ValueError("Nonpositive ratio parts inside log.")
    ln_val = np.log(A) - np.log(B)
    ln_p   = wA/A - wB/B
    ln_pq  = -np.outer(wA,wA)/(A*A) + np.outer(wB,wB)/(B*B)
    return ln_val, ln_p, ln_pq

def ln_Xquad_derivs(r, n):
    """X_r = n_r/Nquad. Return derivatives of ln X_r."""
    n = asDT(n)
    n_r = n[r]
    N   = np.sum(n, dtype=DT)
    if n_r <= 0 or N <= 0:
        raise ValueError("n_r or Nquad nonpositive.")
    e = np.zeros(M, dtype=DT); e[r] = DT("1.0")
    ln_val = np.log(n_r) - np.log(N)
    ln_p   = e/n_r - ONES/N
    ln_pq  = -np.outer(e,e)/(n_r*n_r) + np.outer(ONES,ONES)/(N*N)
    return ln_val, ln_p, ln_pq

# =========================
# MQMQA IDEAL TERM: G_id and H_id
# =========================
def compute_state_id(n):
    n = asDT(n)
    if np.any(n <= 0):
        raise ValueError("All quadruplet moles must be > 0 (logs).")

    Nquad = np.sum(n, dtype=DT)

    # site moles & site fractions
    n_site1 = {i: np.dot(w_site1[i], n) for i in cations}
    n_site2 = {k: np.dot(w_site2[k], n) for k in anions}
    N1 = sum(n_site1.values())
    N2 = sum(n_site2.values())
    X_site1 = {i: n_site1[i]/N1 for i in cations}
    X_site2 = {k: n_site2[k]/N2 for k in anions}

    # pair moles & pair fractions
    n_pair = {(i,k): np.dot(w_pair[(i,k)], n) for i in cations for k in anions}
    Npair  = sum(n_pair.values())
    X_pair = {(i,k): n_pair[(i,k)]/Npair for i in cations for k in anions}

    # row/col sums and F factors
    A_row = {i: sum(n_pair[(i,k)] for k in anions) for i in cations}
    B_col = {k: sum(n_pair[(i,k)] for i in cations) for k in anions}
    F_i = {i: A_row[i]/Npair for i in cations}
    F_k = {k: B_col[k]/Npair for k in anions}

    # quadruplet fractions
    X_quad = n / Nquad

    # site-equivalent fractions in Eq.16 Y-block: Y_i^(1st)=n_i/Nquad, Y_k^(2nd)=n_k/Nquad
    Y1 = {i: n_site1[i]/Nquad for i in cations}
    Y2 = {k: n_site2[k]/Nquad for k in anions}

    return dict(
        Nquad=Nquad, Npair=Npair,
        n_site1=n_site1, n_site2=n_site2,
        X_site1=X_site1, X_site2=X_site2,
        n_pair=n_pair, X_pair=X_pair,
        A_row=A_row, B_col=B_col, F_i=F_i, F_k=F_k,
        X_quad=X_quad, Y1=Y1, Y2=Y2
    )

def G_id(n):
    # Toy MQMQA ideal term (Eq.16): G_id = R T (S1 + S2 + S3).
    st = compute_state_id(n)

    # S1 = sum_sites n ln X (both sublattices)
    S1 = DT("0.0")
    for i in cations:
        S1 += st["n_site1"][i] * np.log(st["X_site1"][i])
    for k in anions:
        S1 += st["n_site2"][k] * np.log(st["X_site2"][k])

    # S2 = sum_pairs n_{i/k} ln( X_{i/k} / (F_i F_k) )
    # Use expanded log: ln(n_{i/k}) + ln(Npair) - ln(A_i) - ln(B_k)
    S2 = DT("0.0")
    Npair = st["Npair"]
    for i in cations:
        Ai = st["A_row"][i]
        for k in anions:
            nik = st["n_pair"][(i,k)]
            Bk  = st["B_col"][k]
            S2 += nik * (np.log(nik) + np.log(Npair) - np.log(Ai) - np.log(Bk))

    # S3 = sum_quad n_r ln u_r
    S3 = DT("0.0")
    Nquad = st["Nquad"]
    for r,(i,j,k,l) in enumerate(quad):
        nr = n[r]
        Xr = nr / Nquad
        C  = (DT("2.0") - (DT("1.0") if i==j else DT("0.0"))) * (DT("2.0") - (DT("1.0") if k==l else DT("0.0")))
        ln_u = np.log(Xr) - np.log(C)

        # -phi * ln of 4 pair fractions
        pairs = [(i,k),(i,l),(j,k),(j,l)]
        ln_u -= phi * sum(np.log(st["X_pair"][pk]) for pk in pairs)

        # +psi * ln of 4 Y factors: (i,j) on 1st sublattice and (k,l) on 2nd
        ln_u += psi * (
            np.log(st["Y1"][i]) + np.log(st["Y1"][j]) +
            np.log(st["Y2"][k]) + np.log(st["Y2"][l])
        )
        S3 += nr * ln_u

    return R*T*(S1 + S2 + S3)

def H_id_analytic(n):
    # Analytic Hessian of the above G_id via n*ln(u) assembly rules
    n = asDT(n)
    compute_state_id(n)  # just to verify positivity

    H = np.zeros((M,M), dtype=DT)

    # ln Npair derivatives (Npair = w_Npair·n; in this toy w_Npair == ONES)
    _, lnNpair_p, lnNpair_pq = ln_linear_derivs(w_Npair, n)

    # ln row/col derivatives
    lnRow_p  = {}
    lnRow_pq = {}
    for i in cations:
        _, lp, lpq = ln_linear_derivs(w_row[i], n)
        lnRow_p[i], lnRow_pq[i] = lp, lpq

    lnCol_p  = {}
    lnCol_pq = {}
    for k in anions:
        _, lp, lpq = ln_linear_derivs(w_col[k], n)
        lnCol_p[k], lnCol_pq[k] = lp, lpq

    # ln pair-mole derivatives
    lnPair_p  = {}
    lnPair_pq = {}
    for i in cations:
        for k in anions:
            _, lp, lpq = ln_linear_derivs(w_pair[(i,k)], n)
            lnPair_p[(i,k)], lnPair_pq[(i,k)] = lp, lpq

    # ---- S1 contributions: n_i ln(n_i/Nquad) for both sublattices ----
    for i in cations:
        wN = w_site1[i]
        Ni = np.dot(wN, n)
        _, ln_p, ln_pq = ln_ratio_derivs(w_site1[i], ONES, n)   # ln(n_i/Nquad)
        H += np.outer(wN, ln_p) + np.outer(ln_p, wN) + Ni*ln_pq

    for k in anions:
        wN = w_site2[k]
        Nk = np.dot(wN, n)
        _, ln_p, ln_pq = ln_ratio_derivs(w_site2[k], ONES, n)   # ln(n_k/Nquad)
        H += np.outer(wN, ln_p) + np.outer(ln_p, wN) + Nk*ln_pq

    # ---- S2 contributions ----
    for i in cations:
        for k in anions:
            wN  = w_pair[(i,k)]
            Nik = np.dot(wN, n)

            # ln u_{i/k} = ln n_{i/k} + ln Npair - ln A_i - ln B_k
            ln_u_p  = lnPair_p[(i,k)]  + lnNpair_p  - lnRow_p[i]  - lnCol_p[k]
            ln_u_pq = lnPair_pq[(i,k)] + lnNpair_pq - lnRow_pq[i] - lnCol_pq[k]

            H += np.outer(wN, ln_u_p) + np.outer(ln_u_p, wN) + Nik*ln_u_pq

    # ---- S3 contributions ----
    # ln X_pair = ln n_{i/k} - ln Npair
    lnXpair_p  = {(i,k): lnPair_p[(i,k)]  - lnNpair_p  for i in cations for k in anions}
    lnXpair_pq = {(i,k): lnPair_pq[(i,k)] - lnNpair_pq for i in cations for k in anions}

    # ln Y1, ln Y2 as ln(n_site/Nquad) = ln_ratio_derivs(w_site, ONES)
    lnY1_p  = {i: ln_ratio_derivs(w_site1[i], ONES, n)[1] for i in cations}
    lnY1_pq = {i: ln_ratio_derivs(w_site1[i], ONES, n)[2] for i in cations}
    lnY2_p  = {k: ln_ratio_derivs(w_site2[k], ONES, n)[1] for k in anions}
    lnY2_pq = {k: ln_ratio_derivs(w_site2[k], ONES, n)[2] for k in anions}

    for r,(i,j,k,l) in enumerate(quad):
        nr = n[r]

        _, lnXr_p,  lnXr_pq  = ln_Xquad_derivs(r, n)

        ln_u_p  = lnXr_p.copy()
        ln_u_pq = lnXr_pq.copy()

        pairs = [(i,k),(i,l),(j,k),(j,l)]
        for pk in pairs:
            ln_u_p  -= phi * lnXpair_p[pk]
            ln_u_pq -= phi * lnXpair_pq[pk]

        ln_u_p  += psi * (lnY1_p[i]  + lnY1_p[j]  + lnY2_p[k]  + lnY2_p[l])
        ln_u_pq += psi * (lnY1_pq[i] + lnY1_pq[j] + lnY2_pq[k] + lnY2_pq[l])

        # Hessian of n_r ln u_r:
        H[r,:] += ln_u_p
        H[:,r] += ln_u_p
        H += nr * ln_u_pq

    return R*T*H

# =========================
# MQMQA EXCESS TERM (toy Eq.17 + updated chi weighting Eq.22)
# =========================
def chi_and_derivs_updated(n, anion):
    """
    Updated Eq.22-like weights so /XY contributes:
      for k=X: weights on anion-pairs: XX=1, XY=0.5, YY=0
      for k=Y: YY=1, XY=0.5, XX=0
    chi_k  = (AA contribution with those weights) / (all ij contribution with those weights)
    tchi_k = (BB contribution with those weights) / (same denom)
    """
    n = asDT(n)

    b     = np.zeros(M, dtype=DT)   # denom weights
    a_chi = np.zeros(M, dtype=DT)   # AA numerator weights
    a_t   = np.zeros(M, dtype=DT)   # BB numerator weights

    if anion == "X":
        for ij in ["AA","AB","BB"]:
            b[idx[f"{ij}_XX"]] = 1.0
            b[idx[f"{ij}_XY"]] = 0.5
        a_chi[idx["AA_XX"]] = 1.0
        a_chi[idx["AA_XY"]] = 0.5
        a_t[idx["BB_XX"]] = 1.0
        a_t[idx["BB_XY"]] = 0.5

    elif anion == "Y":
        for ij in ["AA","AB","BB"]:
            b[idx[f"{ij}_YY"]] = 1.0
            b[idx[f"{ij}_XY"]] = 0.5
        a_chi[idx["AA_YY"]] = 1.0
        a_chi[idx["AA_XY"]] = 0.5
        a_t[idx["BB_YY"]] = 1.0
        a_t[idx["BB_XY"]] = 0.5

    else:
        raise ValueError("anion must be 'X' or 'Y'")

    n = asDT(n)
    B = np.dot(b, n)
    if B <= 0:
        raise ValueError("chi denominator B <= 0")

    chi  = np.dot(a_chi, n) / B
    tchi = np.dot(a_t,   n) / B

    # ratio-of-linear sums derivatives:
    chi_p = (a_chi - chi*b) / B
    t_p   = (a_t   - tchi*b) / B

    chi_pq = np.zeros((M,M), dtype=DT)
    t_pq   = np.zeros((M,M), dtype=DT)
    for p in range(M):
        for q in range(M):
            chi_pq[p,q] = -(b[p]*chi_p[q] + b[q]*chi_p[p]) / B
            t_pq[p,q]   = -(b[p]*t_p[q]   + b[q]*t_p[p])   / B

    return chi, tchi, chi_p, t_p, chi_pq, t_pq

def delta_g_and_derivs_ex(n):
    """
    Toy Δg only for anion-diagonal terms:
      Δg_{ij/XX} = g_{ij,XX} * chi_X * tchi_X
      Δg_{ij/YY} = g_{ij,YY} * chi_Y * tchi_Y
    and Δg(/XY) = 0
    """
    n = asDT(n)
    if np.any(n <= 0):
        raise ValueError("All n must be > 0")

    dg    = np.zeros(M, dtype=DT)
    dg_p  = np.zeros((M,M), dtype=DT)
    dg_pq = np.zeros((M,M,M), dtype=DT)

    gcoeff = {
        "AA_XX": 2000.0, "AB_XX": 3000.0, "BB_XX": 1500.0,
        "AA_YY": 2500.0, "AB_YY": 3500.0, "BB_YY": 1200.0,
    }

    chiX,tX,chiX_p,tX_p,chiX_pq,tX_pq = chi_and_derivs_updated(n,"X")
    chiY,tY,chiY_p,tY_p,chiY_pq,tY_pq = chi_and_derivs_updated(n,"Y")

    def fill(name, chi,tchi, chi_p,t_p, chi_pq,t_pq):
        r = idx[name]
        val = gcoeff[name] * chi * tchi
        dg[r] = val

        Lambda_p  = chi_p/chi + t_p/tchi
        Lambda_pq = (chi_pq/chi - np.outer(chi_p,chi_p)/(chi*chi)) \
                  + (t_pq/tchi   - np.outer(t_p,t_p)/(tchi*tchi))

        dg_p[r,:]    = val * Lambda_p
        dg_pq[r,:,:] = val * (np.outer(Lambda_p, Lambda_p) + Lambda_pq)

    for nm in ["AA_XX","AB_XX","BB_XX"]:
        fill(nm, chiX,tX, chiX_p,tX_p, chiX_pq,tX_pq)
    for nm in ["AA_YY","AB_YY","BB_YY"]:
        fill(nm, chiY,tY, chiY_p,tY_p, chiY_pq,tY_pq)

    return dg, dg_p, dg_pq

def P_Q_and_derivs_ex(n):
    """
    Toy Eq.17-like prefactors with Z=1:

    P_{ij/XX} = 0.5*n_{ij/XY},  P_{ij/YY}=0.5*n_{ij/XY}
    Q_{AA/an} = 0.5*n_{AB/an},  Q_{BB/an}=0.5*n_{AB/an}   for an in {XX,XY,YY}
    """
    n = asDT(n)

    P  = np.zeros(M, dtype=DT)
    Q  = np.zeros(M, dtype=DT)
    Pp = np.zeros((M,M), dtype=DT)
    Qp = np.zeros((M,M), dtype=DT)

    # P terms (l=k): XX and YY
    for ij in ["AA","AB","BB"]:
        ixy = idx[f"{ij}_XY"]
        for kk in ["XX","YY"]:
            r = idx[f"{ij}_{kk}"]
            P[r] = 0.5*n[ixy]
            Pp[r,ixy] = 0.5

    # Q terms (j=i): AA and BB for each anion pair
    for an in ["XX","XY","YY"]:
        iAB = idx[f"AB_{an}"]
        for ii in ["AA","BB"]:
            r = idx[f"{ii}_{an}"]
            Q[r] = 0.5*n[iAB]
            Qp[r,iAB] = 0.5

    return P,Q,Pp,Qp

def G_ex(n):
    """Toy MQMQA excess energy with Eq.17 structure: Gex=0.5*(T1+T2+T3)."""
    n = asDT(n)
    dg,_,_ = delta_g_and_derivs_ex(n)
    P,Q,_,_ = P_Q_and_derivs_ex(n)

    T1 = np.dot(n, dg)

    diag_l_eq_k = [idx[v] for v in ["AA_XX","AB_XX","BB_XX","AA_YY","AB_YY","BB_YY"]]
    T2 = np.dot(P[diag_l_eq_k], dg[diag_l_eq_k])

    diag_j_eq_i = [idx[v] for v in ["AA_XX","BB_XX","AA_XY","BB_XY","AA_YY","BB_YY"]]
    T3 = np.dot(Q[diag_j_eq_i], dg[diag_j_eq_i])

    return DT("0.5")*(T1 + T2 + T3)

def H_ex_analytic(n):
    """Analytic Hessian of the above G_ex."""
    n = asDT(n)
    dg, dg_p, dg_pq = delta_g_and_derivs_ex(n)
    P,Q,Pp,Qp = P_Q_and_derivs_ex(n)

    H = np.zeros((M,M), dtype=DT)

    diag_l_eq_k = [idx[v] for v in ["AA_XX","AB_XX","BB_XX","AA_YY","AB_YY","BB_YY"]]
    diag_j_eq_i = [idx[v] for v in ["AA_XX","BB_XX","AA_XY","BB_XY","AA_YY","BB_YY"]]

    for p in range(M):
        for q in range(M):
            term = 0.0

            # T1_pq = dg_{p,q} + dg_{q,p} + sum_r n_r dg_{r,pq}
            term += dg_p[p,q] + dg_p[q,p]
            term += np.dot(n, dg_pq[:,p,q])

            # T2_pq
            for r in diag_l_eq_k:
                term += Pp[r,p]*dg_p[r,q] + Pp[r,q]*dg_p[r,p] + P[r]*dg_pq[r,p,q]

            # T3_pq
            for r in diag_j_eq_i:
                term += Qp[r,p]*dg_p[r,q] + Qp[r,q]*dg_p[r,p] + Q[r]*dg_pq[r,p,q]

            H[p,q] = 0.5*term

    return H

# =========================
# TOTAL: Gtot and Htot
# =========================
def G_tot(n):
    return G_id(n) + G_ex(n)

def H_tot_analytic(n):
    return H_id_analytic(n) + H_ex_analytic(n)

def H_fd_total(n0, h):
    """Central FD Hessian on scalar G_tot."""
    n0 = asDT(n0)
    h = DT(h)
    if np.min(n0) <= h:
        raise ValueError("h too large: n0-h must stay positive.")

    H = np.zeros((M,M), dtype=DT)
    G0 = G_tot(n0)

    # diagonal
    for p in range(M):
        e = np.zeros(M, dtype=DT); e[p] = DT("1.0")
        H[p,p] = (G_tot(n0+h*e) - DT("2.0")*G0 + G_tot(n0-h*e)) / (h*h)

    # off-diagonal
    for p in range(M):
        ep = np.zeros(M, dtype=DT); ep[p] = DT("1.0")
        for q in range(p+1, M):
            eq = np.zeros(M, dtype=DT); eq[q] = DT("1.0")
            Gpp = G_tot(n0+h*ep+h*eq)
            Gpm = G_tot(n0+h*ep-h*eq)
            Gmp = G_tot(n0-h*ep+h*eq)
            Gmm = G_tot(n0-h*ep-h*eq)
            val = (Gpp - Gpm - Gmp + Gmm) / (DT("4.0")*h*h)
            H[p,q] = val
            H[q,p] = val

    return H

def fro_norm(M):
    M = asDT(M)
    return np.sqrt(np.sum(M * M, dtype=DT))

# =========================
# RUN INTEGRATION TESTS
# =========================
def random_n0(seed):
    rng = np.random.default_rng(seed)
    return asDT(0.5 + rng.random(M))  # all positive and not tiny

if __name__ == "__main__":
    print("Variable order:")
    for i,v in enumerate(vars_):
        print(f"  {i:2d}: {v}")

    for seed in SEEDS:
        n0 = random_n0(seed)

        Ha = H_tot_analytic(n0)
        sym_a = fro_norm(Ha - Ha.T) / max(1.0, fro_norm(Ha))

        print("\n" + "="*70)
        print(f"Seed = {seed}")
        print(f"G_id(n0)  = {float(G_id(n0)): .6e}")
        print(f"G_ex(n0)  = {float(G_ex(n0)): .6e}")
        print(f"G_tot(n0) = {float(G_tot(n0)): .6e}")
        print(f"||H_tot analytic||_F = {float(fro_norm(Ha)): .6e}")
        print(f"analytic symmetry err = {float(sym_a):.3e}")

        best = (None, DT("inf"))
        for h in H_LIST:
            if np.min(n0) <= h:
                print(f"h={h:g}  (skipped; would make some n negative)")
                continue
            Hfd = H_fd_total(n0, h)
            rel = fro_norm(Hfd - Ha) / max(1.0, fro_norm(Ha))
            print(f"h={float(h):g}  rel_err={float(rel):.3e}")
            if rel < best[1]:
                best = (h, rel)

        print(f"Best rel_err ~ {float(best[1]):.3e} at h={best[0]}")

Variable order:
   0: AA_XX
   1: AB_XX
   2: BB_XX
   3: AA_XY
   4: AB_XY
   5: BB_XY
   6: AA_YY
   7: AB_YY
   8: BB_YY

Seed = 0
G_id(n0)  = -9.590841e+04
G_ex(n0)  =  1.227763e+03
G_tot(n0) = -9.468065e+04
||H_tot analytic||_F =  2.366322e+04
analytic symmetry err = 0.000e+00
h=0.01  rel_err=5.724e-05
h=0.001  rel_err=5.723e-07
h=0.0001  rel_err=5.624e-09
h=1e-05  rel_err=3.877e-08
h=1e-06  rel_err=4.165e-06
Best rel_err ~ 5.624e-09 at h=0.0001

Seed = 1
G_id(n0)  = -9.926027e+04
G_ex(n0)  =  1.437572e+03
G_tot(n0) = -9.782270e+04
||H_tot analytic||_F =  1.983016e+04
analytic symmetry err = 0.000e+00
h=0.01  rel_err=3.189e-05
h=0.001  rel_err=3.189e-07
h=0.0001  rel_err=3.658e-09
h=1e-05  rel_err=8.109e-08
h=1e-06  rel_err=4.120e-06
Best rel_err ~ 3.658e-09 at h=0.0001
